In [3]:
import warnings
warnings.filterwarnings("ignore")
from mlflow import MlflowClient, set_tracking_uri
import mlflow
from typing import Tuple
from tqdm import tqdm
import pandas as pd
from datetime import datetime, timedelta
import mysql.connector
import pyarrow
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import argparse
import os
from dateutil.relativedelta import relativedelta

from functions_daily import get_cutoff_indices, transform_ts_data_into_features_and_target, train_test_split, ts_into_features_Daily
from train_tsif_model_daily import ts_into_features_Daily, train_daily



In [6]:
exchange = 'BTC-USD'

X_test_only_numeric, X_train_only_numeric, y_test, y_train = ts_into_features_Daily(exchange)

X_test_only_numeric

MySQL DB Connected


100%|██████████| 1/1 [00:02<00:00,  2.03s/it]

2024-05-01 11:13:06.409577


,open_previous_31_day,open_previous_30_day,open_previous_29_day,open_previous_28_day,open_previous_27_day,open_previous_26_day,open_previous_25_day,open_previous_24_day,open_previous_23_day,open_previous_22_day,...,open_previous_10_day,open_previous_9_day,open_previous_8_day,open_previous_7_day,open_previous_6_day,open_previous_5_day,open_previous_4_day,open_previous_3_day,open_previous_2_day,open_previous_1_day
0,71333.0,69705.0,65447.0,65976.0,68516.0,67841.0,68897.0,69363.0,71633.0,69140.0,...,64936.0,66840.0,66409.0,64275.0,64485.0,63751.0,63424.0,63106.0,63839.0,60609.0
1,69705.0,65447.0,65976.0,68516.0,67841.0,68897.0,69363.0,71633.0,69140.0,70576.0,...,66840.0,66409.0,64275.0,64485.0,63751.0,63424.0,63106.0,63839.0,60609.0,58254.0
2,65447.0,65976.0,68516.0,67841.0,68897.0,69363.0,71633.0,69140.0,70576.0,70061.0,...,66409.0,64275.0,64485.0,63751.0,63424.0,63106.0,63839.0,60609.0,58254.0,59122.0
3,65976.0,68516.0,67841.0,68897.0,69363.0,71633.0,69140.0,70576.0,70061.0,67188.0,...,64275.0,64485.0,63751.0,63424.0,63106.0,63839.0,60609.0,58254.0,59122.0,62891.0
4,68516.0,67841.0,68897.0,69363.0,71633.0,69140.0,70576.0,70061.0,67188.0,63836.0,...,64485.0,63751.0,63424.0,63106.0,63839.0,60609.0,58254.0,59122.0,62891.0,63892.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74,66748.0,66007.0,66189.0,66637.0,66491.0,65147.0,64960.0,64838.0,64114.0,64249.0,...,57023.0,56659.0,58239.0,55850.0,56705.0,58034.0,57730.0,57341.0,57909.0,59225.0
75,66007.0,66189.0,66637.0,66491.0,65147.0,64960.0,64838.0,64114.0,64249.0,63173.0,...,56659.0,58239.0,55850.0,56705.0,58034.0,57730.0,57341.0,57909.0,59225.0,60815.0
76,66189.0,66637.0,66491.0,65147.0,64960.0,64838.0,64114.0,64249.0,63173.0,60266.0,...,58239.0,55850.0,56705.0,58034.0,57730.0,57341.0,57909.0,59225.0,60815.0,64784.0
77,66637.0,66491.0,65147.0,64960.0,64838.0,64114.0,64249.0,63173.0,60266.0,61790.0,...,55850.0,56705.0,58034.0,57730.0,57341.0,57909.0,59225.0,60815.0,64784.0,65092.0


In [7]:
#exchange = 'BTC-USD'

#X_test_only_numeric, X_train_only_numeric = daily_data(exchange)


def predict_daily(exchange):
    
    X_test_only_numeric, X_train_only_numeric, y_test, y_train = ts_into_features_Daily(exchange)
    
    mlflow.set_tracking_uri("http://localhost:5000")
    
    
    model_name = f"{exchange}_Daily_Model"
    model_version = "latest"
    model = mlflow.pyfunc.load_model(model_uri=f"models:/{model_name}/{model_version}")
    
    predictions = model.predict(pd.DataFrame(X_test_only_numeric))

    return predictions 

exchange = 'BTC-USD'
predictions = predict_daily(exchange)

MySQL DB Connected


100%|██████████| 1/1 [00:02<00:00,  2.16s/it]

2024-05-01 11:13:50.651227


In [8]:
predictions

array([61297.566, 58959.855, 58640.25 , 62439.242, 64054.26 , 64324.33 ,
       62699.523, 62712.195, 60861.453, 62270.188, 60146.727, 61162.1  ,
       61487.742, 62756.33 , 61410.047, 66524.58 , 65557.31 , 67108.38 ,
       66985.125, 66385.055, 70714.65 , 70076.22 , 69338.15 , 68226.16 ,
       68070.55 , 69253.14 , 68121.164, 68957.52 , 69000.75 , 68070.53 ,
       68554.875, 66825.305, 67796.01 , 67761.76 , 68391.24 , 70377.52 ,
       71357.72 , 70473.16 , 69742.86 , 69378.19 , 69831.36 , 69490.13 ,
       67348.414, 68454.24 , 67166.83 , 66009.836, 66100.414, 66684.625,
       66326.22 , 65136.16 , 64817.406, 64927.86 , 63915.086, 64202.766,
       62972.22 , 60338.242, 61760.48 , 60583.84 , 61371.895, 60632.28 ,
       61075.246, 62688.438, 62818.832, 61923.035, 60292.03 , 56800.85 ,
       56839.246, 57613.223, 55515.668, 56605.992, 58174.766, 57732.477,
       57341.594, 57716.734, 59067.867, 60637.695, 64391.016, 65266.574,
       64002.766], dtype=float32)

In [11]:
from functions_daily import get_cutoff_indices, transform_ts_data_into_features_and_target, train_test_split
from train_tsif_model_daily import ts_into_features_Daily, train_daily



exchanges = ['BTC-USD']

prueba = train_daily(exchanges)





MySQL DB Connected


100%|██████████| 1/1 [00:01<00:00,  1.96s/it]


2024-05-01 10:50:42.127520


KeyboardInterrupt: 

In [27]:
def ts_into_features_Daily(exchange):
    temporality = 'daily'
    
    connection = mysql.connector.connect(
        user = 'root',
        password = 'root',
        host = 'localhost',
        port = 3306,
        database = 'Historical_Data'
    )
    print("MySQL DB Connected")
    
    cursor = connection.cursor()
    cursor.execute(f"SELECT * FROM FT_DAILY_DATA WHERE Exchange = '{exchange}'")

    results = cursor.fetchall()
    columns = [column[0] for column in cursor.description]

    df_original = pd.DataFrame(results, columns=columns)
    df = df_original[['id_date', 'Open','Exchange']]
    df['datetime'] = pd.to_datetime(df['id_date'], format='%Y%m%d')
    df = df[['datetime', 'Open', 'Exchange']]

    features, targets = transform_ts_data_into_features_and_target(
        df,
        input_seq_len=31*6, # one week of history -> 24*7*1
        step_size=31,
    )
    
    df = pd.concat([features, targets],
               axis = 1)
    
    print(df)
    
    #X_train, y_train, X_test, y_test = train_test_split(
    #    df,
    #    cutoff_date=datetime(2023, 5, 1, 0, 0, 0),
    #    target_column_name='target_open_next_day'
    #)
    
    # Calculate the cutoff_date as the first day of 6 months ago
    cutoff_date = (datetime.now() - relativedelta(months=6)).replace(day=1)
    
    print(cutoff_date)

    # Use the provided train_test_split function
    X_train, y_train, X_test, y_test = train_test_split(
        df,
        cutoff_date=cutoff_date,
        target_column_name='target_open_next_day'
    )
    
    # use only past close data
    past_close_columns = [c for c in X_train.columns if c.startswith('open_')]
    X_train_only_numeric = X_train[past_close_columns]
    X_test_only_numeric = X_test[past_close_columns]


    return X_test_only_numeric, X_train_only_numeric



In [28]:
def predict_daily(exchange):
    
    X_test_only_numeric, X_train_only_numeric = ts_into_features_Daily(exchange)
    
    mlflow.set_tracking_uri("http://localhost:5000")
    
    
    model_name = f"{exchange}_Daily_Model"
    model_version = "latest"
    model = mlflow.pyfunc.load_model(model_uri=f"models:/{model_name}/{model_version}")
    
    predictions = model.predict(pd.DataFrame(X_test_only_numeric))

    return predictions 

exchange = 'BTC-USD'
predictions = predict_daily(exchange)

In [29]:
exchange = 'BTC-USD'
predictions = predict_daily(exchange)

MySQL DB Connected


100%|██████████| 1/1 [00:00<00:00, 12.70it/s]

     open_previous_186_day  open_previous_185_day  open_previous_184_day  \
0                    466.0                  457.0                  424.0   
1                    384.0                  391.0                  389.0   
2                    388.0                  374.0                  380.0   
3                    311.0                  318.0                  330.0   
4                    211.0                  213.0                  211.0   
..                     ...                    ...                    ...   
105                29169.0                28700.0                26636.0   
106                26606.0                26568.0                26533.0   
107                28522.0                28414.0                28332.0   
108                36165.0                36625.0                36586.0   
109                41348.0                42642.0                42261.0   

     open_previous_183_day  open_previous_182_day  open_previous_181_day  \
0          

In [26]:
predictions

array([40169.62 , 54632.04 , 68063.44 , 58678.1  , 74119.375, 63882.02 ],
      dtype=float32)

In [48]:
def ts_into_features_Daily(exchange):
    temporality = 'daily'
    
    connection = mysql.connector.connect(
        user = 'root',
        password = 'root',
        host = 'localhost',
        port = 3306,
        database = 'Historical_Data'
    )
    print("MySQL DB Connected")
    
    cursor = connection.cursor()
    cursor.execute(f"SELECT * FROM FT_DAILY_DATA WHERE Exchange = '{exchange}'")

    results = cursor.fetchall()
    columns = [column[0] for column in cursor.description]

    df_original = pd.DataFrame(results, columns=columns)
    df = df_original[['id_date', 'Open','Exchange']]
    df['datetime'] = pd.to_datetime(df['id_date'], format='%Y%m%d')
    df = df[['datetime', 'Open', 'Exchange']]

    features, targets = transform_ts_data_into_features_and_target(
        df,
        input_seq_len=31, # one week of history -> 24*7*1
        step_size=1,
    )
    
    df = pd.concat([features, targets],
               axis = 1)
    



    return df


In [49]:
exchange = 'BTC-USD'
df = ts_into_features_Daily(exchange)

MySQL DB Connected


100%|██████████| 1/1 [00:02<00:00,  2.77s/it]


In [50]:
df

,open_previous_31_day,open_previous_30_day,open_previous_29_day,open_previous_28_day,open_previous_27_day,open_previous_26_day,open_previous_25_day,open_previous_24_day,open_previous_23_day,open_previous_22_day,...,open_previous_7_day,open_previous_6_day,open_previous_5_day,open_previous_4_day,open_previous_3_day,open_previous_2_day,open_previous_1_day,datetime,exchange,target_open_next_day
0,466.0,457.0,424.0,395.0,408.0,399.0,402.0,436.0,423.0,411.0,...,361.0,363.0,378.0,392.0,401.0,395.0,383.0,2014-10-18,BTC-USD,384.0
1,457.0,424.0,395.0,408.0,399.0,402.0,436.0,423.0,411.0,404.0,...,363.0,378.0,392.0,401.0,395.0,383.0,384.0,2014-10-19,BTC-USD,391.0
2,424.0,395.0,408.0,399.0,402.0,436.0,423.0,411.0,404.0,399.0,...,378.0,392.0,401.0,395.0,383.0,384.0,391.0,2014-10-20,BTC-USD,389.0
3,395.0,408.0,399.0,402.0,436.0,423.0,411.0,404.0,399.0,377.0,...,392.0,401.0,395.0,383.0,384.0,391.0,389.0,2014-10-21,BTC-USD,382.0
4,408.0,399.0,402.0,436.0,423.0,411.0,404.0,399.0,377.0,376.0,...,401.0,395.0,383.0,384.0,391.0,389.0,382.0,2014-10-22,BTC-USD,386.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3557,68243.0,66748.0,66007.0,66189.0,66637.0,66491.0,65147.0,64960.0,64838.0,64114.0,...,58239.0,55850.0,56705.0,58034.0,57730.0,57341.0,57909.0,2024-07-14,BTC-USD,59225.0
3558,66748.0,66007.0,66189.0,66637.0,66491.0,65147.0,64960.0,64838.0,64114.0,64249.0,...,55850.0,56705.0,58034.0,57730.0,57341.0,57909.0,59225.0,2024-07-15,BTC-USD,60815.0
3559,66007.0,66189.0,66637.0,66491.0,65147.0,64960.0,64838.0,64114.0,64249.0,63173.0,...,56705.0,58034.0,57730.0,57341.0,57909.0,59225.0,60815.0,2024-07-16,BTC-USD,64784.0
3560,66189.0,66637.0,66491.0,65147.0,64960.0,64838.0,64114.0,64249.0,63173.0,60266.0,...,58034.0,57730.0,57341.0,57909.0,59225.0,60815.0,64784.0,2024-07-17,BTC-USD,65092.0
